# Idea

This notebook splits the dataset for training and testing using stratified group k-fold (stratified by label distribution + unique msisdn).
This step is done separately in this notebook to preserve the split versions of the datasets (Manual data version control).

The datasets generated from this notebook will be saved in data/dvc/

Note: this notebook was used to version datasets v1 and v2, also dataset v3 was versioned from v2 in the feature engineering stage.

---

### Imports


In [1]:
import pandas as pd

from pathlib import Path

from sklearn.model_selection import StratifiedGroupKFold

### Paths

In [ ]:
PROJECT_ROOT = Path.cwd().parent.parent

DATA_PATH = PROJECT_ROOT / "data"

PROFILE_PATH = DATA_PATH / "profiles" / "all_profiles.parquet"

DVC_PATH = DATA_PATH / "dvc"
TRAIN_V2_PATH = DVC_PATH / "train_v2.parquet"
TEST_V2_PATH = DVC_PATH / "test_v2.parquet"

---
### Load dataset

In [3]:
profiles_df = pd.read_parquet(PROFILE_PATH)

print(f"Loaded {len(profiles_df):,} rows.")

profiles_df.head()

Loaded 428,800 rows.


,msisdn,label,outgoing_calls,incoming_calls,avg_outgoing_duration,avg_incoming_duration,unique_callers,unique_callees,incoming_outgoing_ratio,caller_incoming_ratio,...,afternoon_ratio,evening_calls,evening_ratio,night_calls,night_ratio,deep_night_calls,deep_night_ratio,period_entropy,avg_delay,profile_date
0,22362444330,0,0,5,0.0,136.200000,4,0,999999.0,0.8,...,0.200000,2,0.4,0,0.0,0,0.0,1.521928,5413.750000,2022-11-01
1,22667898029,0,0,1,0.0,90.000000,1,0,999999.0,1.0,...,1.000000,0,0.0,0,0.0,0,0.0,-0.000000,0.000000,2022-11-01
2,22995468560,0,0,2,0.0,38.000000,1,0,999999.0,0.5,...,0.000000,0,0.0,0,0.0,0,0.0,-0.000000,37.000000,2022-11-01
3,2250101000087,0,2,3,57.5,23.666667,3,2,1.5,1.0,...,0.000000,2,0.4,0,0.0,0,0.0,0.970951,5077.250000,2022-11-01
4,2250101000108,0,10,2,329.1,40.000000,2,9,0.2,1.0,...,0.583333,0,0.0,0,0.0,0,0.0,1.384432,2799.818182,2022-11-01


### Verify dataset

In [4]:
print("\nLabel Distribution:")

print(
    profiles_df["label"]
    .value_counts(normalize=True)
    .round(4)
)

print(
    f"\nUnique MSISDNs: "
    f"{profiles_df['msisdn'].nunique():,}"
)

print(
    f"Unique Days: "
    f"{profiles_df['profile_date'].nunique()}"
)

display(
    profiles_df["profile_date"]
    .value_counts()
    .sort_index()
)


Label Distribution:
label
0    0.9425
1    0.0575
Name: proportion, dtype: float64

Unique MSISDNs: 31,742
Unique Days: 15


profile_date
2022-11-01    28004
2022-11-02    28401
2022-11-03    28501
2022-11-04    28719
2022-11-05    28648
2022-11-06    28468
2022-11-07    28916
2022-11-08    28771
2022-11-09    28761
2022-11-10    28535
2022-11-11    28886
2022-11-12    28755
2022-11-13    28446
2022-11-14    28584
2022-11-15    28405
Name: count, dtype: int64

---
### Train/test split

In [5]:
sgkf = StratifiedGroupKFold(
    n_splits=5,
    shuffle=True,
    random_state=42
)

X = profiles_df.drop(
    columns="label"
)

y = profiles_df["label"]

groups = profiles_df["msisdn"]

train_idx, test_idx = next(
    sgkf.split(
        X,
        y,
        groups
    )
)

train_df = (
    profiles_df
    .iloc[train_idx]
    .reset_index(drop=True)
)

test_df = (
    profiles_df
    .iloc[test_idx]
    .reset_index(drop=True)
)

print(
    f"Train Rows: {len(train_df):,}"
)

print(
    f"Test Rows: {len(test_df):,}"
)

Train Rows: 343,040
Test Rows: 85,760


### Verify split quality

In [6]:
overlap = (
    set(train_df["msisdn"])
    &
    set(test_df["msisdn"])
)

print(f"MSISDN Leakage: {len(overlap)}")

# ----------- Train -----------
print("\nTrain Label Distribution:")

print(
    train_df["label"]
    .value_counts(normalize=True)
    .round(4)
)

# ------------ Test ------------
print("\nTest Label Distribution:")

print(
    test_df["label"]
    .value_counts(normalize=True)
    .round(4)
)

print(
    f"\nTrain MSISDNs: "
    f"{train_df['msisdn'].nunique():,}"
)

print(
    f"Test MSISDNs: "
    f"{test_df['msisdn'].nunique():,}"
)

MSISDN Leakage: 0

Train Label Distribution:
label
0    0.9425
1    0.0575
Name: proportion, dtype: float64

Test Label Distribution:
label
0    0.9425
1    0.0575
Name: proportion, dtype: float64

Train MSISDNs: 25,394
Test MSISDNs: 6,348


---
### Save versioned datasets

In [7]:
train_df.to_parquet(
    TRAIN_V2_PATH,
    index=False
)

test_df.to_parquet(
    TEST_V2_PATH,
    index=False
)

print("Saved:")
print(TRAIN_V2_PATH)
print(TEST_V2_PATH)

Saved:
c:\Users\ali.farhat\Documents\simbox-project\data\dvc\train_v2.parquet
c:\Users\ali.farhat\Documents\simbox-project\data\dvc\test_v2.parquet


---
### Version metadata

In [9]:
print("=" * 50)
print("DATASET VERSION: V2")
print("=" * 50)

print(
    f"Source Dataset: "
    f"{PROFILE_PATH.name}"
)

print(
    f"Total Rows: "
    f"{len(profiles_df):,}"
)

print(
    f"Train Rows: "
    f"{len(train_df):,}"
)

print(
    f"Test Rows: "
    f"{len(test_df):,}"
)

print(
    f"Unique MSISDNs: "
    f"{profiles_df['msisdn'].nunique():,}"
)

print(
    f"Train MSISDNs: "
    f"{train_df['msisdn'].nunique():,}"
)

print(
    f"Test MSISDNs: "
    f"{test_df['msisdn'].nunique():,}"
)

print(
    f"MSISDN Leakage: "
    f"{len(overlap)}"
)

print(
    f"Unique Days: "
    f"{profiles_df['profile_date'].nunique()}"
)

print(
    f"Date Range: "
    f"{profiles_df['profile_date'].min()} "
    f"to "
    f"{profiles_df['profile_date'].max()}"
)

print(
    "\nTrain Label Distribution:"
)

print(
    train_df["label"]
    .value_counts(normalize=True)
    .round(4)
)

print(
    "\nTest Label Distribution:"
)

print(
    test_df["label"]
    .value_counts(normalize=True)
    .round(4)
)

DATASET VERSION: V2
Source Dataset: all_profiles.parquet
Total Rows: 428,800
Train Rows: 343,040
Test Rows: 85,760
Unique MSISDNs: 31,742
Train MSISDNs: 25,394
Test MSISDNs: 6,348
MSISDN Leakage: 0
Unique Days: 15
Date Range: 2022-11-01 to 2022-11-15

Train Label Distribution:
label
0    0.9425
1    0.0575
Name: proportion, dtype: float64

Test Label Distribution:
label
0    0.9425
1    0.0575
Name: proportion, dtype: float64
